In [1]:
!pip install -q -U transformers peft accelerate bitsandbytes
!pip uninstall -q -y torchao
!pip install -q fastapi uvicorn hf_transfer
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('Kurulum tamam')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 122.0 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.5 MB/s eta 0:00:00a 0:00:01
Kurulum tamam


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/bitirme-ui')

QWEN_CACHE = BASE_DIR / 'qwen_cache' / 'Qwen2.5-7B-Instruct'
QWEN_BASE = str(QWEN_CACHE) if QWEN_CACHE.is_dir() else 'Qwen/Qwen2.5-7B-Instruct'

ADAPTERS = {
    'tr': BASE_DIR / 'sentiment' / 'models' / 'qwen_tr_final',
    'en': BASE_DIR / 'sentiment' / 'models' / 'qwen_en_final',
}

print(f'Base model : {QWEN_BASE}')
for lang, p in ADAPTERS.items():
    ok = (p / 'adapter_model.safetensors').exists()
    print(f'Adapter {lang} : {p} -> {"OK" if ok else "BULUNAMADI!"}')
    assert ok, f'Adapter eksik: {p}'

Mounted at /content/drive
Base model : /content/drive/MyDrive/bitirme-ui/qwen_cache/Qwen2.5-7B-Instruct
Adapter tr : /content/drive/MyDrive/bitirme-ui/sentiment/models/qwen_tr_final -> OK
Adapter en : /content/drive/MyDrive/bitirme-ui/sentiment/models/qwen_en_final -> OK


In [3]:
import json, threading

import torch
import torch.nn.functional as F
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          BitsAndBytesConfig)
from peft import PeftModel

assert torch.cuda.is_available(), 'GPU runtime seç (Runtime -> Change runtime type -> T4)'

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'GPU: {torch.cuda.get_device_name(0)} | compute dtype: {COMPUTE_DTYPE}')

LABELS = {lang: json.loads((p / 'label_classes.json').read_text())
          for lang, p in ADAPTERS.items()}

TOKENIZERS = {}
for lang, p in ADAPTERS.items():
    tok = AutoTokenizer.from_pretrained(str(p), trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    TOKENIZERS[lang] = tok

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)
base = AutoModelForSequenceClassification.from_pretrained(
    QWEN_BASE, num_labels=len(LABELS['tr']),
    quantization_config=bnb, device_map='auto', trust_remote_code=True,
)
base.config.pad_token_id = TOKENIZERS['tr'].pad_token_id

model = PeftModel.from_pretrained(base, str(ADAPTERS['tr']), adapter_name='tr')
model.load_adapter(str(ADAPTERS['en']), adapter_name='en')
model.eval()

_LOCK = threading.Lock() 

@torch.no_grad()
def predict(text: str, lang: str):
    tok = TOKENIZERS[lang]
    labels = LABELS[lang]
    with _LOCK:
        model.set_adapter(lang)
        model.config.pad_token_id = tok.pad_token_id
        enc = tok(text, return_tensors='pt', truncation=True, max_length=256).to('cuda')
        with torch.amp.autocast('cuda', dtype=COMPUTE_DTYPE):
            logits = model(**enc).logits
    probs = F.softmax(logits.float(), dim=-1).squeeze().cpu().tolist()
    pred = labels[max(range(len(probs)), key=probs.__getitem__)]
    return pred, labels, probs

for lang, text in [('tr', 'Bu film harikaydı, bayıldım!'),
                   ('en', 'This movie was absolutely terrible.')]:
    pred, labels, probs = predict(text, lang)
    top3 = sorted(zip(labels, probs), key=lambda x: -x[1])[:3]
    print(f'[{lang}] {text!r} -> {pred} | {[(l, round(p, 3)) for l, p in top3]}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


GPU: Tesla T4 | compute dtype: torch.bfloat16


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Qwen2ForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/bitirme-ui/qwen_cache/Qwen2.5-7B-Instruct
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


[tr] 'Bu film harikaydı, bayıldım!' -> happiness | [('happiness', 0.988), ('fear', 0.007), ('anger', 0.004)]
[en] 'This movie was absolutely terrible.' -> fear | [('fear', 0.97), ('disgust', 0.026), ('anger', 0.003)]


In [4]:
import time
import requests
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

api = FastAPI(title='Qwen Sentiment API')

class PredictRequest(BaseModel):
    text: str
    lang: str = 'tr'

@api.get('/health')
def health():
    return {'status': 'ok',
            'gpu': torch.cuda.get_device_name(0),
            'langs': list(ADAPTERS)}

@api.post('/predict')
def predict_endpoint(req: PredictRequest):
    if req.lang not in ADAPTERS:
        raise HTTPException(400, f'lang {req.lang!r} not supported (tr/en)')
    if not req.text.strip():
        raise HTTPException(400, 'empty field')
    t0 = time.time()
    label, labels, probs = predict(req.text, req.lang)
    return {'label': label, 'labels': labels, 'probs': probs,
            'lang': req.lang, 'elapsed_ms': int((time.time() - t0) * 1000)}

server = uvicorn.Server(uvicorn.Config(api, host='0.0.0.0', port=8000, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()

for _ in range(30):
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        print('Lokal API ayakta:', r.json()) 
        break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('API baslamadi — bu hücreyi tekrar çalıştır')

Lokal API ayakta: {'status': 'ok', 'gpu': 'Tesla T4', 'langs': ['tr', 'en']}


In [5]:
import re, subprocess

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

PUBLIC_URL = None
deadline = time.time() + 60
while time.time() < deadline and PUBLIC_URL is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        PUBLIC_URL = m.group(0)

assert PUBLIC_URL, 'Tünel URL alınamadı — bu hücreyi tekrar çalıştır'

threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

tunnel_ok = False
for _ in range(15):
    try:
        if requests.get(f'{PUBLIC_URL}/health', timeout=5).ok:
            tunnel_ok = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

print('=' * 62)
print('QWEN API HAZIR — bu URL\'yi Streamlit kenar çubuğuna yapıştır:')
print()
print(f'    {PUBLIC_URL}')
print()
print('=' * 62)

QWEN API HAZIR — bu URL'yi Streamlit kenar çubuğuna yapıştır:

    https://stages-twist-observer-stops.trycloudflare.com

